# NHOS WS Demo

This notebook demonstrates how to connect to the New Horizons Desktop Backend WebUI stream endpoint, subscribe to a specific `device_uid`, and continuously receive `snapshot` and `update` events.

The backend routes are mounted under `/newhorizons`. The stream endpoint is `GET/WS /newhorizons/stream/ws`. Authentication works with `Authorization: Bearer <token>` or `?token=<token>`.

## Prerequisites

- The Desktop Backend is running, usually at `http://127.0.0.1:5051`
- You know a valid `username` and `password`
- You know the `device_uid` you want to subscribe to
- `requests` and `websocket-client` are installed

If an old `socketio` package happens to be installed, do not use it. This backend uses plain WebSocket, not Socket.IO.

In [1]:
# If needed, install the dependencies first:
# %pip install requests websocket-client pandas

import json
from datetime import datetime

import requests
import websocket


In [ ]:
BASE_URL = "https://isensing-s1.u-aizu.ac.jp"
APP_PREFIX = "/newhorizons"
USERNAME = "admin"
PASSWORD = "password"
DEVICE_UID = "3CDC7545CCD0"

API_BASE_URL = BASE_URL.rstrip("/") + APP_PREFIX
WS_URL = BASE_URL.replace("http://", "ws://").replace("https://", "wss://").rstrip("/") + APP_PREFIX + "/stream/ws"
API_BASE_URL, WS_URL

('https://isensing-s1.u-aizu.ac.jp/newhorizons',
 'wss://isensing-s1.u-aizu.ac.jp/newhorizons/stream/ws')

In [4]:
resp = requests.post(
    f"{API_BASE_URL}/api/token",
    json={"username": USERNAME, "password": PASSWORD},
)

if resp.status_code != 200:
    print(f"token request failed: {resp.status_code}")
    print(resp.text)
resp.raise_for_status()

token_payload = resp.json()
token = token_payload["token"]
expires_in = token_payload.get("expires_in")

print("token acquired successfully")
print(f"expires_in: {expires_in}")
print(f"token[:40]: {token[:40]}...")

token acquired successfully
expires_in: 86400
token[:40]: eyJ1c2VybmFtZSI6ImFkbWluIiwicm9sZSI6ImFk...


## Open the stream connection

After the connection opens, the backend sends `hello_ack` first. Then we send `subscribe` for the target `device_uid`. If the subscription succeeds, you usually receive `subscribed`, and if the device already has cached data, you also receive a `snapshot`.

In [5]:
ws = websocket.create_connection(
    WS_URL,
    header=[f"Authorization: Bearer {token}"],
    timeout=10,
)
ws.settimeout(1.0)

hello_ack = json.loads(ws.recv())
hello_ack

{'type': 'hello_ack',
 'server_time': '2026-06-15T04:30:50.855129+00:00',
 'username': 'admin',
 'role': 'admin',
 'transport_mode': 'udp_stream'}

In [6]:
ws.send(json.dumps({"type": "subscribe", "device_uid": DEVICE_UID}, separators=(",", ":")))

subscribed = json.loads(ws.recv())
subscribed

{'type': 'subscribed', 'device_uid': '3CDC7545CCD0'}

## Receive events continuously

The loop below stores each event in `events` and prints it immediately. The event format matches the backend implementation. Common event types include:

- `snapshot`: latest state after subscription
- `update`: incremental updates after the snapshot
- `stream_ended`: the device is no longer available or the subscription was removed
- `error`: message format or authorization error

In [ ]:
events = []

try:
    while True:
        raw = ws.recv()
        event = json.loads(raw)
        events.append({"received_at": datetime.now().isoformat(timespec="seconds"), **event})
        print(event)
except KeyboardInterrupt:
    print("Stream stopped manually")
except websocket.WebSocketTimeoutException:
    print("Timed out waiting for new events")
finally:
    ws.close()
    print("WebSocket closed")

{'type': 'snapshot', 'device_uid': '3CDC7545CCD0', 'data': {'protocol': 'NHO/Arduino/1', 'dn': '3CDC7545CCD0', 'device_uid': '3CDC7545CCD0', 'packet_device_uid': '3CDC7545CCD0', 'packet_version': 3, 'frame_id': 35076, 'timestamp_ms': 665924, 'ts': 665.924, 'sn': 210, 'p': [318.0, 315.0, 319.0, 314.0, 315.0, 317.0, 313.0, 312.0, 315.0, 317.0, 315.0, 315.0, 312.0, 317.0, 309.0, 315.0, 314.0, 315.0, 317.0, 321.0, 315.0, 316.0, 315.0, 317.0, 316.0, 319.0, 317.0, 317.0, 318.0, 315.0, 315.0, 315.0, 315.0, 315.0, 317.0, 317.0, 319.0, 317.0, 317.0, 315.0, 317.0, 315.0, 313.0, 315.0, 316.0, 317.0, 317.0, 317.0, 315.0, 317.0, 315.0, 317.0, 315.0, 315.0, 314.0, 315.0, 317.0, 314.0, 313.0, 313.0, 317.0, 315.0, 317.0, 316.0, 315.0, 315.0, 315.0, 314.0, 316.0, 315.0, 315.0, 315.0, 315.0, 316.0, 315.0, 317.0, 315.0, 315.0, 315.0, 315.0, 315.0, 315.0, 315.0, 317.0, 315.0, 316.0, 314.0, 316.0, 314.0, 315.0, 315.0, 317.0, 315.0, 314.0, 317.0, 314.0, 317.0, 314.0, 314.0, 314.0, 317.0, 314.0, 315.0, 317.0

## Convert to a table

If you want to do more analysis, convert `events` into a `pandas.DataFrame`. For `snapshot` and `update` events, the payload is usually under the `data` field.

In [ ]:
try:
    import pandas as pd

    df = pd.DataFrame(events)
    df.head()
except ImportError:
    print("pandas is not installed, skipping DataFrame conversion")

## Notes

If you need to connect to a remote environment, change `BASE_URL` to the actual host address and keep the `/newhorizons` prefix. If you are using HTTPS, the notebook automatically switches to `wss://`. If no data arrives, first confirm that `DEVICE_UID` is correct and that the backend has a usable visualization snapshot for that device.